## Setup

This notebook calls language models through OpenRouter, a service that gives one API to many different models.

The first code cell loads your `OPENROUTER_API_KEY` from a `.env` file, or asks you to type it in. Keep your key private: do not paste it into shared notebooks or commit it to a shared repository.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

### What the imports do

- `create_agent`: builds an agent, which is a model that can use memory and tools.
- `SummarizationMiddleware`: a plug-in that checks how big the conversation history is and summarises it when it gets too large.
- `RunnableConfig`: a small settings dictionary we use to pass the `thread_id` (explained below).
- `ChatOpenRouter`: connects our code to a chat model through OpenRouter.
- `InMemorySaver`: a checkpointer, which stores the conversation so the agent can read it on the next call.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.runnables import RunnableConfig
from langchain_openrouter import ChatOpenRouter
from langgraph.checkpoint.memory import InMemorySaver

/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


# Memory in LangChain

[Short-term Memory](https://docs.langchain.com/oss/python/langchain/short-term-memory)

<img src="https://www.gettingstarted.ai/content/images/size/w2400/2023/10/memory_diagram-0627c68230aa438f9b5419064d63cbbc.png">


### Reading & Writing: The Two Memory Must-Dos ✍️

1. **Reading**: Memory feeds context into the model's input each time it responds.
2. **Writing**: Memory saves the latest session details for next time.

## What is short-term memory?

A language model has no memory of its own. Each time you send a message, the application must send the model everything it needs to see, usually the instructions plus the earlier messages in the conversation. The application is responsible for remembering the conversation. The model only "remembers" what is in its input for that one call.

Short-term memory lets your application remember previous interactions within a single **thread**, which is one conversation.

Conversation history is the most common form of short-term memory.

### Why this matters: the context window

The amount of text a model can read in one call is its **context window**. It is measured in **tokens**, which are small pieces of words.

A long conversation can fill the context window. Even when a model accepts a long input, it can get "distracted" by old or off-topic content, respond more slowly, and cost more to run.

*Key idea: a model only knows what is inside its context window.*

*We saw how to add memory (a checkpointer) in the previous notebook on agents.*

Chat models take input as **messages**: a system message (the instructions) and human messages (what the user says). In a chat, messages alternate between the user and the model, so the list grows over time. Because the window is limited, many applications deliberately "forget" old information. Common solutions are:

1. [Trim messages](https://docs.langchain.com/oss/python/langchain/short-term-memory#trim-messages)
2. [Delete messages](https://docs.langchain.com/oss/python/langchain/short-term-memory#delete-messages)
3. Custom strategies, such as filtering messages

A very common solution is to **summarise** the older messages and keep the summary as context.

## Example of summarisation

Trimming or deleting messages is simple, but you can lose useful information when old messages are removed. A more careful approach is to use a chat model to **summarise** the message history, so the key facts are kept in shorter form.

![Diagram of message summarisation](https://mintcdn.com/langchain-5e9cc07a/ybiAaBfoBvFquMDz/oss/images/summary.png?w=1100&fit=max&auto=format&n=ybiAaBfoBvFquMDz&q=85&s=4abdac693a562788aa0db8681bef8ea7)

The demo below uses very low numbers so you can see the summary appear quickly:

- **Trigger** at 80 tokens: when the history reaches this size, summarising starts.
- **Keep** the last 2 messages: the older messages are replaced by the summary.

In a real app you would use something like 4000 tokens and keep 20 messages.

## Building the agent

The code uses **two models**:

- **Summariser** (`nvidia/nemotron-3.5-lightning:free`): a small model whose only job is to write the summary.
- **Main agent** (`openai/gpt-6-luna`): the model that answers you.

The code comment describes the summariser as free. Pricing and availability can change, so check the provider's page before relying on either model.

The `create_agent` call combines these pieces:

- `model`: the main agent model.
- `middleware`: the `SummarizationMiddleware` plug-in, with the summariser model, a `trigger` and a `keep` setting.
- `checkpointer`: `InMemorySaver`, which stores the conversation.

The `trigger` and `keep` settings are written as pairs:

- `trigger=("tokens", 80)`: start summarising when the history reaches 80 tokens.
- `keep=("messages", 2)`: after summarising, keep the 2 most recent messages as they are.

The loop then sends four messages one after another, using the same `thread_id`, so they all belong to one conversation. The final question, "what's my name?", tests whether the agent still remembers.

In [3]:
# Free Nemotron does the cheap summarising, Luna is the main agent
model_nano = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free")
model_mini = ChatOpenRouter(model="openai/gpt-6-luna")

checkpointer = InMemorySaver()

agent = create_agent(
    model=model_mini,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model_nano,
            trigger=("tokens", 80),  # Trigger summarization at 80 tokens (use ~4000 in real apps)
            keep=("messages", 2),     # Keep last 2 messages after summary (use ~20 in real apps)
        )
    ],
    checkpointer=checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}
for text in [
    "hi, my name is Mo",
    "write a short poem about cats",
    "now do the same but for dogs",
    "what's my name?",
]:
    response = agent.invoke({"messages": text}, config)
    response["messages"][-1].pretty_print()

================================== Ai Message ==================================

Hi Mo, nice to meet you! How can I help today?
================================== Ai Message ==================================

Soft paws tread where moonbeams lie,  
Bright eyes watch the stars drift by.  
A purring loaf, a whiskered face—  
Cats make any home their place.
================================== Ai Message ==================================

Soft paws trot through morning light,  
Happy tails wag left and right.  
A loyal friend, a joyful bark—  
Dogs bring sunshine to the park.
================================== Ai Message ==================================

I don’t know—you haven’t told me your name.


## Looking inside the memory

The code below prints each message stored for this thread.

The first line is a **summary message**. It replaces the older messages. The lines after it are the most recent messages, kept word for word.

A summary keeps the important facts in shorter form. Here, the name "Mo" survived because the summary included it. That is a good result, but it is not a guarantee. A summary is written by a model, so it can drop or change details. When summaries are used in real apps, important facts should be checked.

In [4]:
# Inspect the history: the older messages should be replaced by a summary message
for m in agent.get_state(config).values["messages"]:
    print(type(m).__name__, "->", str(m.content)[:120].replace("\n", " "))

HumanMessage -> Here is a summary of the conversation to date:  ## SESSION INTENT The user requests a short poem about dogs, mirroring t
AIMessage -> Soft paws trot through morning light,   Happy tails wag left and right.   A loyal friend, a joyful bark—   Dogs bring su
HumanMessage -> what's my name?
AIMessage -> I don’t know—you haven’t told me your name.


### Threads are isolated

A **thread** is one conversation, identified by its `thread_id`. Think of it as a separate notebook for each customer: each one has its own history, and nothing written in one is visible in another.

A new thread starts with an empty history. The next cell tests this by asking the same question on a new thread, `"2"`.

In [5]:
other_config: RunnableConfig = {"configurable": {"thread_id": "2"}}
agent.invoke({"messages": "what's my name?"}, other_config)["messages"][-1].pretty_print()

================================== Ai Message ==================================

I don’t know—you haven’t told me your name in this chat.


### Persistence

`InMemorySaver` keeps memory in the computer's RAM. When the notebook kernel restarts, the memory is lost. This is fine for learning, but real apps need memory that survives restarts.

LangGraph provides other saver classes for this, such as:

- `SqliteSaver`: stores the memory in a local SQLite database file.
- `PostgresSaver`: stores the memory in a PostgreSQL database.

Their setup differs from `InMemorySaver`, so check the [LangChain persistence documentation](https://docs.langchain.com/oss/python/langgraph/persistence) before using them.

## Try it yourself

1. Change `trigger` to a larger number, or `keep` to a different number. What changes in the history output?
2. Tell the agent a fact early in a conversation, then send several longer messages before asking for it again. Does it still remember? Why or why not?
3. **Discuss:** Imagine a summary drops a detail such as a medical allergy or a deadline. What could go wrong, and how would you design an app to reduce that risk?